In [1]:
import pandas as pd

# =========================
# 1. File names
# =========================
file1 = "Ablation1_Gpt.xlsx"
file2 = "Ablation1_Claude.xlsx"

# =========================
# 2. Read Excel files
# =========================
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)

# =========================
# 3. Clean column names
# =========================
df1.columns = df1.columns.str.strip()
df2.columns = df2.columns.str.strip()

# Rename second column to Condition
df1 = df1.rename(columns={"Unnamed: 1": "Condition"})
df2 = df2.rename(columns={"Unnamed: 1": "Condition"})

# =========================
# 4. Remove empty rows
# =========================
df1 = df1[df1["ID"].notna()].copy()
df2 = df2[df2["ID"].notna()].copy()

# Clean ID and Condition
df1["ID"] = df1["ID"].astype(str).str.strip()
df2["ID"] = df2["ID"].astype(str).str.strip()

df1["Condition"] = df1["Condition"].astype(str).str.strip()
df2["Condition"] = df2["Condition"].astype(str).str.strip()

# =========================
# 5. Columns to average
# =========================
score_cols = ["HRO_Our", "TCF_Our", "PPA_Our", "MA_Our", "SPQ_Our", "CSDC_Our", "CER_Our", "Total"]

# Make sure numeric
for col in score_cols:
    df1[col] = pd.to_numeric(df1[col], errors="coerce")
    df2[col] = pd.to_numeric(df2[col], errors="coerce")

# =========================
# 6. Merge on ID + Condition
# =========================
merged = pd.merge(
    df1[["ID", "Condition"] + score_cols],
    df2[["ID", "Condition"] + score_cols],
    on=["ID", "Condition"],
    suffixes=("_file1", "_file2")
)

# =========================
# 7. Average each score column
# =========================
avg_df = merged[["ID", "Condition"]].copy()

for col in score_cols:
    avg_df[col] = (merged[f"{col}_file1"] + merged[f"{col}_file2"]) / 2

# Optional: round values
avg_df[score_cols] = avg_df[score_cols].round(2)

# =========================
# 8. Sort nicely
# =========================
condition_order = {"Full": 1, "Only L1": 2, "Only L2": 3, "Baseline": 4}
avg_df["condition_sort"] = avg_df["Condition"].map(condition_order)

# Sort ID numerically like R2, R5, R10
avg_df["id_num"] = avg_df["ID"].str.extract(r"(\d+)").astype(int)

avg_df = avg_df.sort_values(["id_num", "condition_sort"]).drop(columns=["condition_sort", "id_num"])

# =========================
# 9. Show result
# =========================
print(avg_df)

# =========================
# 10. Save to Excel
# =========================
output_file = "Averaged_Ablation.xlsx"
avg_df.to_excel(output_file, index=False)

print(f"\nSaved as: {output_file}")

     ID Condition  HRO_Our  TCF_Our  PPA_Our  MA_Our  SPQ_Our  CSDC_Our  \
28   R5      Full     23.0     12.5     13.5     9.5      9.0      13.0   
29   R5   Only L1     21.0     11.0     12.0     7.0      7.5      15.0   
30   R5   Only L2     15.5      9.0      8.5     6.0      7.0      12.0   
31   R5  Baseline     17.0     11.5     12.0     6.0      7.0      12.0   
8   R10      Full     24.5     13.0     15.0     9.5      9.5      15.0   
9   R10   Only L1     22.0     14.5     14.0     8.0      9.0      14.0   
10  R10   Only L2     19.5     11.0     11.0     6.5      6.5      13.0   
11  R10  Baseline     16.5      9.5      9.0     5.5      6.5       9.0   
20  R12      Full     22.5     13.5     12.0     9.0     10.0      13.5   
21  R12   Only L1     20.5     12.0     13.0     7.5      8.5      12.5   
22  R12   Only L2     19.0     11.5     12.0     7.0      7.0      12.0   
23  R12  Baseline     18.5      9.5     11.5     6.5      6.5      10.5   
36  R17      Full     23.

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel

# =========================
# 1. Load averaged file (from previous step)
# =========================
df = pd.read_excel("Averaged_Ablation.xlsx")

# =========================
# 2. Define conditions
# =========================
FULL = "Full"
L1 = "Only L1"
L2 = "Only L2"
BASE = "Baseline"

# =========================
# 3. Metrics
# =========================
metrics = ["HRO_Our", "TCF_Our", "PPA_Our", "MA_Our", "SPQ_Our", "CSDC_Our", "CER_Our", "Total"]

# =========================
# 4. Split data by condition
# =========================
df_full = df[df["Condition"] == FULL].set_index("ID")
df_l1 = df[df["Condition"] == L1].set_index("ID")
df_l2 = df[df["Condition"] == L2].set_index("ID")
df_base = df[df["Condition"] == BASE].set_index("ID")

# Align IDs
df_full, df_l1 = df_full.align(df_l1, join="inner", axis=0)
df_full, df_l2 = df_full.align(df_l2, join="inner", axis=0)
df_full, df_base = df_full.align(df_base, join="inner", axis=0)

# =========================
# 5. Function: Cohen's d (paired)
# =========================
def cohens_d_paired(x, y):
    diff = x - y
    return diff.mean() / diff.std(ddof=1)

# =========================
# 6. Compute means
# =========================
mean_full = df_full[metrics].mean()
mean_l1 = df_l1[metrics].mean()
mean_l2 = df_l2[metrics].mean()
mean_base = df_base[metrics].mean()

# =========================
# 7. Compute drops
# =========================
drop_l1 = mean_full["Total"] - mean_l1["Total"]
drop_l2 = mean_full["Total"] - mean_l2["Total"]
drop_base = mean_full["Total"] - mean_base["Total"]

# =========================
# 8. Paired t-tests (Total only)
# =========================
t_l1, p_l1 = ttest_rel(df_full["Total"], df_l1["Total"])
t_l2, p_l2 = ttest_rel(df_full["Total"], df_l2["Total"])
t_base, p_base = ttest_rel(df_full["Total"], df_base["Total"])

# =========================
# 9. Cohen's d
# =========================
d_l1 = cohens_d_paired(df_full["Total"], df_l1["Total"])
d_l2 = cohens_d_paired(df_full["Total"], df_l2["Total"])
d_base = cohens_d_paired(df_full["Total"], df_base["Total"])

# =========================
# 10. Build final table
# =========================
table = pd.DataFrame([
    ["A3 Full Fusion (Ours)", *mean_full.values, "-", "-", "-"],
    ["A1 L1 Only (LLM)", *mean_l1.values, round(drop_l1,2), round(p_l1,4), round(d_l1,2)],
    ["A2 L2 Only (Corpus)", *mean_l2.values, round(drop_l2,2), round(p_l2,4), round(d_l2,2)],
    ["A4 Baseline", *mean_base.values, round(drop_base,2), round(p_base,4), round(d_base,2)]
],
columns=["Condition"] + metrics + ["Drop", "p-value", "Cohen's d"])

# =========================
# 11. Round values nicely
# =========================
for col in metrics:
    table[col] = table[col].astype(float).round(2)

# =========================
# 12. Show result
# =========================
print("\n===== FINAL ABLATION TABLE =====\n")
print(table)

# =========================
# 13. Save as Excel
# =========================
table.to_excel("Final_Ablation_Table.xlsx", index=False)
print("\nSaved as Final_Ablation_Table.xlsx")


===== FINAL ABLATION TABLE =====

               Condition  HRO_Our  TCF_Our  PPA_Our  MA_Our  SPQ_Our  \
0  A3 Full Fusion (Ours)    23.85    12.50    14.35    9.55     9.70   
1       A1 L1 Only (LLM)    21.50    11.90    13.10    7.75     8.45   
2    A2 L2 Only (Corpus)    19.10    11.85    10.45    7.50     7.50   
3            A4 Baseline    16.85    10.00    10.00    6.10     6.35   

   CSDC_Our  CER_Our  Total  Drop p-value Cohen's d  
0     14.50     9.75   94.2     -       -         -  
1     13.60     8.70   85.0   9.2     0.0      2.79  
2     12.45     7.75   76.6  17.6     0.0       2.8  
3      9.90     4.70   63.9  30.3     0.0      5.92  

Saved as Final_Ablation_Table.xlsx


In [4]:
from IPython.display import display

display(table)

,Condition,HRO_Our,TCF_Our,PPA_Our,MA_Our,SPQ_Our,CSDC_Our,CER_Our,Total,Drop,p-value,Cohen's d
0,A3 Full Fusion (Ours),23.85,12.50,14.35,9.55,9.70,14.50,9.75,94.20,-,-,-
1,A1 L1 Only (LLM),21.50,11.90,13.10,7.75,8.45,13.60,8.70,85.00,9.20,0.00,2.79
2,A2 L2 Only (Corpus),19.10,11.85,10.45,7.50,7.50,12.45,7.75,76.60,17.60,0.00,2.80
3,A4 Baseline,16.85,10.00,10.00,6.10,6.35,9.90,4.70,63.90,30.30,0.00,5.92


In [3]:
import pandas as pd

# Show ALL columns
pd.set_option('display.max_columns', None)

# Prevent line wrapping
pd.set_option('display.width', 1000)

# Show full column content
pd.set_option('display.max_colwidth', None)

# Optional: make float formatting clean
pd.set_option('display.float_format', '{:.2f}'.format)

# Now print again
print("\n===== FINAL ABLATION TABLE =====\n")
print(table)


===== FINAL ABLATION TABLE =====

               Condition  HRO_Our  TCF_Our  PPA_Our  MA_Our  SPQ_Our  CSDC_Our  CER_Our  Total  Drop p-value Cohen's d
0  A3 Full Fusion (Ours)    23.85    12.50    14.35    9.55     9.70     14.50     9.75  94.20     -       -         -
1       A1 L1 Only (LLM)    21.50    11.90    13.10    7.75     8.45     13.60     8.70  85.00  9.20    0.00      2.79
2    A2 L2 Only (Corpus)    19.10    11.85    10.45    7.50     7.50     12.45     7.75  76.60 17.60    0.00      2.80
3            A4 Baseline    16.85    10.00    10.00    6.10     6.35      9.90     4.70  63.90 30.30    0.00      5.92
